In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
from tqdm import tqdm
import pandas as pd
import os
import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import random

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Set environment variables
os.environ["CUDA_VISIBLE_DEVICES"] = '0'


PyTorch version: 2.7.1+cu128
CUDA available: True
GPU device: NVIDIA GeForce RTX 5070 Ti
CUDA version: 12.8
Using device: cuda:0


In [ ]:
def ReadExcel(excelpath: str):
    """
    Read the Excel file, skipping the first row and the first column (ID).
    Returns: numpy array with shape (number of samples, 190)
             The first 180 columns are inputs: 60 time steps × 3 features = [downstream water level boundary, discharge flow 1, discharge flow 2]
             The last 10 columns are outputs: 1 time step × 10 features
    """
    DF = pd.read_excel(excelpath, header=None)  # Do not automatically recognize headers
    # Skip the first row and the first column (ID)
    Values = DF.iloc[1:, 1:].values  # Start from the second row, second column
    # Ensure all data is of type float32
    Values = Values.astype('float32') 
    print(f"File {os.path.basename(excelpath)} shape: {Values.shape}")
    return Values

# Define model parameters
hist_step = 60  # Number of historical time steps
pred_step = 1   # Number of prediction time steps
input_channel = 3  # Number of input features: [downstream water level boundary, discharge flow 1, discharge flow 2]
output_channel = 10 # Number of output features

input_data_num = hist_step * input_channel  # 60 * 3 = 180


In [ ]:
# Base path
base_path = r'D:\0DATA\NHRI\UV'

# Load all 22 condition data
data_files = [
    ("1、23_1200_UV.xlsx", "UV_1_23_1200"),
    ("2、32_300_UV.xlsx", "UV_2_32_300"),
    ("3、32_160_UV.xlsx", "UV_3_32_160"),
    ("4、32_sametime_UV.xlsx", "UV_4_32_sametime"),
    ("5、23_160_UV.xlsx", "UV_5_23_160"),
    ("6、23_300_UV.xlsx", "UV_6_23_300"),
    ("7、23_600_UV.xlsx", "UV_7_23_600"),
    ("8、32_600_UV.xlsx", "UV_8_32_600"),
    ("9、23_900_UV.xlsx", "UV_9_23_900"),
    ("10、32_900_UV.xlsx", "UV_10_32_900"),
    ("11、32_1200_UV.xlsx", "UV_11_32_1200"),
    ("12、23_1500_UV.xlsx", "UV_12_23_1500"),
    ("13、32_1500_UV.xlsx", "UV_13_32_1500"),
    ("14、23_450_UV.xlsx", "UV_14_23_450"),
    ("15、32_450_UV.xlsx", "UV_15_32_450"),
    ("16、23_750_UV.xlsx", "UV_16_23_750"),
    ("17、32_750_UV.xlsx", "UV_17_32_750"),
    ("18、23_1050_UV.xlsx", "UV_18_23_1050"),
    ("19、32_1050_UV.xlsx", "UV_19_32_1050"),
    ("20、23_1350_UV.xlsx", "UV_20_23_1350"),
    ("21、32_1350_UV.xlsx", "UV_21_32_1350"),
    ("22、32_sametime_UV.xlsx", "UV_22_32_sametime")
]

# Create a dictionary to store all data
all_data = {}
for filename, varname in data_files:
    filepath = os.path.join(base_path, filename)
    all_data[varname] = ReadExcel(filepath)

# Check data shapes
print("\nData shapes for each condition:")
for name, data in all_data.items():
    print(f"{name}: {data.shape}")

# Split training and testing data as per user's request
# Test set: 7, 13, 17, 18, 22
test_conditions = ['UV_7_23_600', 'UV_13_32_1500', 'UV_17_32_750', 'UV_18_23_1050', 'UV_22_32_sametime']
train_conditions = [name for name in all_data.keys() if name not in test_conditions]

print(f"\nTraining set conditions ({len(train_conditions)} conditions): {train_conditions}")
print(f"Test set conditions ({len(test_conditions)} conditions): {test_conditions}")


In [ ]:
# Extract training data for fitting the normalizer
def extract_input_matrix(data_dict, condition_names):
    """Extract input data from specified conditions and reshape it to 2D"""
    X_all = np.vstack([data_dict[name][:, :input_data_num] for name in condition_names])
    # Reshape: (N_total, 180) -> (N_total, 60, 3) -> (N_total*60, 3)
    X_reshaped = X_all.reshape(-1, hist_step, input_channel)
    return X_reshaped.reshape(-1, input_channel)

def extract_output_matrix(data_dict, condition_names):
    """Extract output data from specified conditions"""
    return np.vstack([data_dict[name][:, input_data_num:] for name in condition_names])

# Prepare training data for normalization
X_train_for_scaler = extract_input_matrix(all_data, train_conditions)
Y_train_for_scaler = extract_output_matrix(all_data, train_conditions)

print(f"Normalized training input data shape: {X_train_for_scaler.shape}")
print(f"Normalized training output data shape: {Y_train_for_scaler.shape}")

# Create and fit the normalizers
scaler_input = MinMaxScaler()
scaler_output = MinMaxScaler()

scaler_input.fit(X_train_for_scaler)
scaler_output.fit(Y_train_for_scaler)

print("\nNormalizers fitted successfully!")
print(f"Input normalizer: min={scaler_input.data_min_}, scale={scaler_input.scale_}")
print(f"Output normalizer: min={scaler_output.data_min_}, scale={scaler_output.scale_}")


scaler_save_path = r'D:\0DATA\NHRI\Normalizers'
os.makedirs(scaler_save_path, exist_ok=True)

import joblib

# Save normalizers to file
scaler_dict = {
    'scaler_input': scaler_input,
    'scaler_output': scaler_output
}
joblib.dump(scaler_dict, os.path.join(scaler_save_path, 'scalers.pkl'))
print(f"\nNormalizers saved to: {os.path.join(scaler_save_path, 'scalers.pkl')}")

# Also save each normalizer separately for easier use
joblib.dump(scaler_input, os.path.join(scaler_save_path, 'scaler_input.pkl'))
joblib.dump(scaler_output, os.path.join(scaler_save_path, 'scaler_output.pkl'))

# Save normalizer parameters (if manual recovery is needed)
scaler_params_path = os.path.join(scaler_save_path, 'scaler_parameters.npz')
np.savez(scaler_params_path,
         input_min=scaler_input.data_min_,
         input_max=scaler_input.data_max_,
         output_min=scaler_output.data_min_,
         output_max=scaler_output.data_max_)
print(f"Normalizer parameters saved to: {scaler_params_path}")


# Normalization function
def normalize_condition_data(data_dict, condition_names):
    """Normalize data for specified conditions"""
    normalized = []
    for name in condition_names:
        data = data_dict[name]
        # Input part: first 180 columns
        x = data[:, :input_data_num].reshape(-1, hist_step, input_channel)
        x_norm = scaler_input.transform(x.reshape(-1, input_channel)).reshape(-1, hist_step, input_channel)
        # Output part: last 10 columns
        y_norm = scaler_output.transform(data[:, input_data_num:])
        normalized.append((x_norm, y_norm))
    return normalized

# Normalize training and test data
train_data = normalize_condition_data(all_data, train_conditions)
test_data = normalize_condition_data(all_data, test_conditions)

print(f"\nNumber of normalized training conditions: {len(train_data)}")
print(f"Number of normalized test conditions: {len(test_data)}")
if len(train_data) > 0:
    print(f"Single training condition input shape: {train_data[0][0].shape}, output shape: {train_data[0][1].shape}")


In [ ]:
class ConditionDataset(Dataset):
    def __init__(self, x_array, y_array):
        self.X = torch.tensor(x_array, dtype=torch.float32)  # (N, 60, 3)
        self.Y = torch.tensor(y_array, dtype=torch.float32)  # (N, 10)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def build_data_loaders(data_list, batch_size):
    """Create DataLoader for each condition"""
    loaders = []
    for x, y in data_list:
        ds = ConditionDataset(x, y)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False)
        loaders.append(dl)
    return loaders

# Set batch size
BatchSize = 256

# Create data loaders
train_loaders = build_data_loaders(train_data, batch_size=BatchSize)
test_loaders = build_data_loaders(test_data, batch_size=BatchSize)

print(f"Number of DataLoaders in training set: {len(train_loaders)}")
print(f"Number of DataLoaders in test set: {len(test_loaders)}")
if len(train_loaders) > 0:
    sample_x, sample_y = next(iter(train_loaders[0]))
    print(f"Shape of a single batch input: {sample_x.shape}, output shape: {sample_y.shape}")


In [ ]:
def median_absolute_error(y_true, y_pred):
    """Median Absolute Error (MdAE)"""
    errors = np.abs(y_true - y_pred)
    return np.median(errors)

def iqr_based_mae(y_true, y_pred):
    """
    IQR-based MAE: Calculate the MAE for samples whose absolute errors fall within the [25%, 75%] range
    """
    errors = np.abs(y_true - y_pred).flatten()
    q1 = np.percentile(errors, 25)
    q3 = np.percentile(errors, 75)
    mask = (errors >= q1) & (errors <= q3)
    filtered_errors = errors[mask]
    if len(filtered_errors) == 0:
        return np.nan
    return np.mean(filtered_errors)

def hit_rate(y_true, y_pred, delta):
    """
    Hit rate calculation based on the maximum true value
    y_true, y_pred: [T, num_features]
    delta: tolerance threshold (percentage, e.g., 5 means 5%)
    Returns hit rate (%)
    """
    max_val = np.max(np.abs(y_true))
    absolute_errors = np.abs(y_pred - y_true)
    hits = absolute_errors <= (delta / 100.0) * max_val
    return np.mean(hits) * 100

def safe_mape(y_true, y_pred, epsilon=1e-8, threshold=0.01):
    """
    Enhanced MAPE calculation, handling zero and near-zero values
    Returns MAPE value (in percentage)
    """
    # Create mask: ignore points near zero
    mask = np.abs(y_true) > threshold
    # If no valid points, return NaN
    if mask.sum() == 0:
        return np.nan
    # Calculate absolute percentage error (only for valid points)
    ape = np.abs((y_true[mask] - y_pred[mask]) / (y_true[mask] + epsilon))
    return np.mean(ape) * 100


In [ ]:
from model.Informer import Model as Informer
import torch
import torch.nn as nn
import time

class Config:
    def __init__(self):
        self.seq_len = 60
        self.pred_len = 1  # Predict only one step
        self.d_model = 256
        self.d_state = 16
        self.d_ff = 256
        self.e_layers = 2
        self.d_layers = 1
        self.dropout = 0.32
        self.embed = 'timeF'
        self.freq = 'h'
        self.activation = 'gelu'
        self.output_attention = False
        self.use_norm = False
        #self.class_strategy = 'cls'
        self.channel_independence = False
        self.enc_in = 3      # Number of input features
        self.dec_in = 3      # Decoder is no longer used, but still needs configuration
        self.c_out = 10      # Number of output features
        self.factor = 1
        self.n_heads = 8
        self.distil = True
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create model configuration
args = Config()
args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("args.device = ", args.device)
model = Informer(args).to(args.device)
print(model)

print(f"Model device: {args.device}")
print(f"Model configuration:")
print(f"  Input features (enc_in): {args.enc_in}")
print(f"  Output features (c_out): {args.c_out}")
print(f"  Sequence length (seq_len): {args.seq_len}")
print(f"  Prediction length (pred_len): {args.pred_len}")
print(f"  Model parameters count: {sum(p.numel() for p in model.parameters())}")


In [ ]:
# Training configuration
num_epochs = 10000
epsilon = 0.0001
max_grad_norm = 1.0
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, eps=epsilon, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, 
                                                     patience=5, threshold=1e-4, cooldown=2, min_lr=1e-6)

# Save directory
save_dir = r'D:\0DATA\NHRI\ModelFiles'
os.makedirs(save_dir, exist_ok=True)

# Training records
train_loss_history = []
test_loss_history = []
test_horizontal_mape_history = []  # Horizontal flow MAPE history
test_vertical_mape_history = []    # Vertical flow MAPE history
test_horizontal_rmse_history = []  # Horizontal flow RMSE history
test_vertical_rmse_history = []    # Vertical flow RMSE history
test_horizontal_me_history = []    # Horizontal flow maximum error history
test_vertical_me_history = []      # Vertical flow maximum error history
test_horizontal_iqr_mae_history = []  # Horizontal flow IQR_MAE history
test_vertical_iqr_mae_history = []    # Vertical flow IQR_MAE history
best_test_loss = float('inf')
best_V_ME = float('inf')

# Early stopping variables
early_stopping_patience = 200
early_stopping_counter = 0
best_model_state = None
best_epoch = 0

def calculate_rmse(y_true, y_pred):
    """Calculate RMSE"""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def calculate_max_error(y_true, y_pred):
    """Calculate maximum error"""
    return np.max(np.abs(y_true - y_pred))

def calculate_metrics_horizontal_vertical(y_true, y_pred):
    """
    Calculate metrics for horizontal and vertical flow velocities separately
    Assumes output order: horizontal1, vertical1, horizontal2, vertical2, ..., horizontal5, vertical5
    """
    # Extract horizontal flow (values at positions 1, 3, 5, 7, 9)
    horizontal_true = y_true[:, [0, 2, 4, 6, 8]]
    horizontal_pred = y_pred[:, [0, 2, 4, 6, 8]]
    
    # Extract vertical flow (values at positions 2, 4, 6, 8, 10)
    vertical_true = y_true[:, [1, 3, 5, 7, 9]]
    vertical_pred = y_pred[:, [1, 3, 5, 7, 9]]
    
    # Calculate horizontal metrics
    horizontal_mape = safe_mape(horizontal_true, horizontal_pred)
    horizontal_rmse = calculate_rmse(horizontal_true, horizontal_pred)
    horizontal_me = calculate_max_error(horizontal_true, horizontal_pred)
    horizontal_iqr_mae = iqr_based_mae(horizontal_true, horizontal_pred)
    
    # Calculate vertical metrics
    vertical_mape = safe_mape(vertical_true, vertical_pred)
    vertical_rmse = calculate_rmse(vertical_true, vertical_pred)
    vertical_me = calculate_max_error(vertical_true, vertical_pred)
    vertical_iqr_mae = iqr_based_mae(vertical_true, vertical_pred)
    
    return (horizontal_mape, horizontal_rmse, horizontal_me, horizontal_iqr_mae,
            vertical_mape, vertical_rmse, vertical_me, vertical_iqr_mae)

start_time = time.time()

print("Training started...")
for epoch in range(num_epochs):
    # Training phase
    model.train()
    epoch_train_losses = []
    
    for loader in train_loaders:
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            
            B, L, D = x.shape
            # Create time stamps
            x_mark_enc = torch.zeros(B, L, 4).to(device)
            x_dec = torch.zeros(B, args.pred_len, D).to(device)
            x_mark_dec = torch.zeros(B, args.pred_len, 4).to(device)
            
            optimizer.zero_grad()
            out = model(x, x_mark_enc, x_dec, x_mark_dec)
            out = out[:, -1, :]  # [B, 10]
            loss = criterion(out, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            
            epoch_train_losses.append(loss.item())
    
    # Validation phase (using original test set)
    model.eval()
    epoch_test_losses = []
    epoch_test_preds = []
    epoch_test_targets = []
    
    with torch.no_grad():
        for loader in test_loaders:
            for x, y in loader:
                x = x.to(device)
                y = y.to(device)
                
                B, L, D = x.shape
                x_mark_enc = torch.zeros(B, L, 4).to(device)
                x_dec = torch.zeros(B, args.pred_len, D).to(device)
                x_mark_dec = torch.zeros(B, args.pred_len, 4).to(device)
                
                out = model(x, x_mark_enc, x_dec, x_mark_dec)
                out = out[:, -1, :]
                loss = criterion(out, y)
                
                epoch_test_losses.append(loss.item())
                epoch_test_preds.append(out.cpu())
                epoch_test_targets.append(y.cpu())
    
    # Calculate average loss
    avg_train_loss = np.mean(epoch_train_losses) if epoch_train_losses else 0
    avg_test_loss = np.mean(epoch_test_losses) if epoch_test_losses else 0
    
    # Calculate metrics
    if epoch_test_preds:
        test_preds = torch.cat(epoch_test_preds, dim=0).numpy()
        test_targets = torch.cat(epoch_test_targets, dim=0).numpy()
        
        # Inverse transform
        test_preds_inv = scaler_output.inverse_transform(test_preds)
        test_targets_inv = scaler_output.inverse_transform(test_targets)
        
        # Calculate horizontal and vertical metrics
        (horizontal_mape, horizontal_rmse, horizontal_me, horizontal_iqr_mae,
         vertical_mape, vertical_rmse, vertical_me, vertical_iqr_mae) = calculate_metrics_horizontal_vertical(
            test_targets_inv, test_preds_inv
        )
    else:
        horizontal_mape = horizontal_rmse = horizontal_me = horizontal_iqr_mae = np.nan
        vertical_mape = vertical_rmse = vertical_me = vertical_iqr_mae = np.nan
    
    # Record history
    train_loss_history.append(avg_train_loss)
    test_loss_history.append(avg_test_loss)
    test_horizontal_mape_history.append(horizontal_mape)
    test_vertical_mape_history.append(vertical_mape)
    test_horizontal_rmse_history.append(horizontal_rmse)
    test_vertical_rmse_history.append(vertical_rmse)
    test_horizontal_me_history.append(horizontal_me)
    test_vertical_me_history.append(vertical_me)
    test_horizontal_iqr_mae_history.append(horizontal_iqr_mae)
    test_vertical_iqr_mae_history.append(vertical_iqr_mae)
    
    # Learning rate scheduling
    scheduler.step(avg_test_loss)
    
    # Update the best model and early stopping logic
    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        best_epoch = epoch
        best_model_state = model.state_dict().copy()  # Save best model state
        early_stopping_counter = 0  # Reset early stopping counter
        save_reason = "Best test loss"
        save_model = True
    else:
        early_stopping_counter += 1
        save_model = False
    
    # Save the model with the smallest vertical flow maximum error:
    if vertical_me < best_V_ME:
        save_model = True
        save_reason = "Smallest vertical flow maximum error"
        best_V_ME = vertical_me

    # Save periodically
    if epoch % 100 == 0:
        save_model = True
        save_reason = "Periodic save"
    
    # Save model
    if save_model:
        model_path = os.path.join(save_dir, f'epoch.{epoch:05d}-test_loss.{avg_test_loss:.5f}-hMAPE.{horizontal_mape:.5f}-vMAPE.{vertical_mape:.5f}.pth')
        torch.save(model.state_dict(), model_path)
    
    # Print progress
    if epoch % 10 == 0 or save_model:
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch:4d}/{num_epochs}], Train Loss: {avg_train_loss:.5f}, Test Loss: {avg_test_loss:.5f}, LR: {current_lr:.6f}')
        print(f'  Horizontal flow - MAPE: {horizontal_mape:.5f}%, RMSE: {horizontal_rmse:.5f}, ME: {horizontal_me:.5f}, IQR_MAE: {horizontal_iqr_mae:.5f}')
        print(f'  Vertical flow - MAPE: {vertical_mape:.5f}%, RMSE: {vertical_rmse:.5f}, ME: {vertical_me:.5f}, IQR_MAE: {vertical_iqr_mae:.5f}')
        if save_model:
            print(f'  [Model saved: {save_reason}]')
        print(f'  Early stopping counter: {early_stopping_counter}/{early_stopping_patience}')
        print('-' * 80)
    
    # Check for early stopping condition
    if early_stopping_counter >= early_stopping_patience:
        print(f'\nEarly stopping triggered! Training stopped after {epoch} epochs.')
        print(f'Best test loss {best_test_loss:.5f} achieved at epoch {best_epoch}.')
        break

# Save the best model after training
if best_model_state is not None:
    # Save the best model
    best_model_path = os.path.join(save_dir, f'best_model-epoch.{best_epoch:05d}-test_loss.{best_test_loss:.5f}.pth')
    torch.save(best_model_state, best_model_path)
    print(f"Best model saved to: {best_model_path}")
    
    # Load best model parameters into the current model
    model.load_state_dict(best_model_state)

# Save the final model
final_model_path = os.path.join(save_dir, f'final_epoch.{epoch:05d}-test_loss.{avg_test_loss:.5f}-hMAPE.{horizontal_mape:.5f}-vMAPE.{vertical_mape:.5f}.pth')
torch.save(model.state_dict(), final_model_path)

training_time = time.time() - start_time
print(f"\nTraining completed! Total time: {training_time:.2f} seconds")

# Save the training process
csv_path = r'D:\0DATA\NHRI\training_process.csv'
np.savetxt(csv_path, np.column_stack((
    train_loss_history, 
    test_loss_history, 
    test_horizontal_mape_history,
    test_vertical_mape_history,
    test_horizontal_rmse_history,
    test_vertical_rmse_history,
    test_horizontal_me_history,
    test_vertical_me_history,
    test_horizontal_iqr_mae_history,
    test_vertical_iqr_mae_history
)), delimiter=",", 
header="Training Loss, Test Loss, Horizontal MAPE, Vertical MAPE, Horizontal RMSE, Vertical RMSE, Horizontal ME, Vertical ME, Horizontal IQR_MAE, Vertical IQR_MAE", comments='')
print(f"Training process saved to: {csv_path}")
